# 06 — Comparative results, statistics and systems analysis

All numbers below come from `results/final_summary.json` and the generated tables. The common test
set has 1,800 samples; thresholds were selected on validation only. No test-set tuning was
performed.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
DATA = ROOT / "data"
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def show_image(path):
    """Render a saved PNG inside the notebook without extra display dependencies."""
    img = PILImage.open(path)
    plt.figure(figsize=(min(12, img.width / 100), min(8, img.height / 100)))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.close()


print("Project root:", ROOT)

## 1. Master comparison

In [ ]:
master = pd.read_csv(RESULTS / "tables" / "master_comparison.csv")
cols = ["model_label", "encoding", "n_features", "accuracy", "precision", "recall", "f1",
        "roc_auc", "fpr", "fnr", "qubits", "circuit_depth"]
master[cols].round(4)

In [ ]:
show_image(RESULTS / "figures" / "model_comparison.png")
show_image(RESULTS / "figures" / "roc_comparison.png")

CNN+LSTM leads on accuracy (0.9567) and F1 (0.9533). QSVM leads on recall (0.9737) and has a
ROC-AUC 0.0012 higher, but with a 0.1185 false-positive rate. VQC trails on every headline metric.

## 2. Cross-validation

In [ ]:
cv = pd.read_csv(RESULTS / "tables" / "cross_validation_summary.csv")
cv[cv["metric"] == "f1"][["model", "n_folds", "mean", "std", "ci_low", "ci_high"]].round(4)

In [ ]:
show_image(RESULTS / "figures" / "cross_validation_f1.png")

Three-fold CV F1: CNN 0.9346 ± 0.0135, QSVM 0.9180 ± 0.0161, VQC 0.7452 ± 0.0527. The ordering is
stable across folds. Three folds are underpowered for a Wilcoxon test, which is stated explicitly.

## 3. Paired statistical tests

In [ ]:
stats = json.loads((RESULTS / "metrics" / "statistical_tests.json").read_text(encoding="utf-8"))
pd.DataFrame(stats["mcnemar"])[["comparison", "statistic", "p_value", "n_discordant",
                                 "effect_size_cohens_h", "effect_interpretation"]].round(6)

In [ ]:
pd.DataFrame(stats["paired_bootstrap"])[["comparison", "metric", "difference", "ci_low",
                                          "ci_high", "crosses_zero"]].round(4)

In [ ]:
pd.DataFrame(stats["wilcoxon"])[["comparison", "n_folds", "p_value", "significant"]]

McNemar rejects equal error patterns for all three pairs (CNN vs QSVM p = 2.389e-6). The paired
bootstrap puts the CNN−QSVM F1 difference at 0.0302 [0.0166, 0.0434], while the AUC difference
−0.0012 [−0.0061, 0.0030] crosses zero. With three folds the Wilcoxon test cannot reach
significance at α = 0.05; it is reported as supporting evidence only.

## 4. Unseen attack categories and external novel types

In [ ]:
unseen = pd.read_csv(RESULTS / "tables" / "unseen_category_results.csv")
unseen[["model", "threshold", "accuracy", "precision", "recall", "f1", "fpr", "roc_auc"]].round(4)

In [ ]:
external = pd.read_csv(RESULTS / "tables" / "external_novel_attack_results.csv")
external[["model", "threshold", "accuracy", "precision", "recall", "f1", "fpr", "mcc", "roc_auc"]].round(4)

In [ ]:
show_image(RESULTS / "figures" / "unseen_attack_performance.png")

On the balanced R2L+U2R holdout, VQC detects 76.2% of held-out attacks versus 4.8% for CNN and
1.2% for QSVM. This is a small single-seed result and is **not** evidence of general quantum
superiority. On the external KDDTest+ novel types, QSVM F1 is 0.6572 (ROC-AUC 0.8917), VQC is
0.5189 (0.7801) and CNN is 0.4451 (0.6665); the CNN has the lowest FPR (0.0807). The QSVM's
external lead is a scope-specific result on one normal-heavy file, not a general quantum advantage.

## 5. Latency and quantum resources

In [ ]:
lat = pd.read_csv(RESULTS / "tables" / "latency_comparison.csv")
lat[["model", "inference_ms_per_sample", "end_to_end_ms_per_sample",
     "throughput_samples_per_s", "slowdown_vs_classical"]].round(3)

In [ ]:
qres = pd.read_csv(RESULTS / "tables" / "quantum_resources.csv")
qres[["model_label", "qubits", "circuit_depth", "circuit_size", "num_two_qubit_gates",
      "num_parameters", "optimizer"]]

In [ ]:
show_image(RESULTS / "figures" / "latency_comparison.png")
show_image(RESULTS / "figures" / "quantum_resources.png")

Single-sample CPU medians: CNN 0.92 ms, QSVM 3.94 ms (4.28×), VQC 32.73 ms (35.51×). These are
simulator timings on one machine, not hardware or deployment claims. Note that the evaluation table
reports batch throughput separately; the latency table is the single-sample benchmark.

## 6. Explainability, drift and alerting

In [ ]:
show_image(RESULTS / "figures" / "permutation_importance.png")
show_image(RESULTS / "explainability" / "shap_summary.png")

In [ ]:
drift = pd.read_csv(RESULTS / "tables" / "drift_summary.csv")
drift.round(4)

In [ ]:
alerts = json.loads((RESULTS / "metrics" / "alert_summary.json").read_text(encoding="utf-8"))
pd.Series(alerts["level_counts"]).to_frame("samples")

Permutation importance, PCA attribution, SHAP and quantum input sensitivity are all generated from
the trained models. The drift experiment is intentionally limited to the classical model and shows
that feature permutation is far more damaging than additive Gaussian noise. The alert engine maps
calibrated probabilities to five tiers; it is assistive and does not auto-block below ESCALATE.

## 7. Final summary

In [ ]:
summary = json.loads((RESULTS / "final_summary.json").read_text(encoding="utf-8"))
print("Mode:", summary["mode"], "| seed:", summary["seed"])
print("Leakage audit passed:", summary["leakage_audit_passed"])
print("Common test size:", summary["class_distribution"]["test"]["rows"])
print("Selected features:", summary["feature_selection"]["method"], summary["feature_selection"]["n_features"])

**Final conclusion.** Under identical inputs, splits and evaluation, the classical CNN+LSTM is the
strongest model (F1 0.9533). QSVM is competitive and has a marginally higher but unresolved
ROC-AUC. VQC is weaker and budget-limited. No quantum advantage is claimed; all quantum results are
exact statevector simulations.